# Charts Reference

Comprehensive reference for all diagnostic and evaluation charts in the bid-euchre project.

## Two Chart Families

| Module | Input Type | Use Case |
|--------|------------|----------|
| `bid_euchre.diagnostics` | **DataFrame** | Interactive dataset analysis, health checks |
| `bid_euchre.reporting` | **Dict/List** | Training pipeline reports, batch generation |

Both have functions with the same names but different APIs — choose based on your data format.

In [ ]:
# Auto-reload for development
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import shutil
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image

from bid_euchre.diagnostics import (
    plot_feature_correlation,
    plot_feature_distributions,
    plot_hand_value_by_contract,
    plot_hand_value_by_seat,
    plot_rolling_mean,
)
from bid_euchre.diagnostics.charts import plot_feature_vs_label
from bid_euchre.features.hand_eval import get_hand_features
from bid_euchre.reporting.validation import (
    generate_validation_plots,
)
from bid_euchre.reporting.validation import (
    plot_feature_correlation as eval_plot_feature_correlation,
)
from bid_euchre.reporting.validation import (
    plot_feature_distributions as eval_plot_feature_distributions,
)
from bid_euchre.reporting.validation import (
    plot_hand_value_by_contract as eval_plot_hand_value_by_contract,
)
from bid_euchre.sim.deals import generate_deal

plt.style.use("seaborn-v0_8-whitegrid")

---
## Sample Data Generation

Generate sample datasets in both formats for demonstration.

In [ ]:
# Generate DataFrame for diagnostic charts
SEED = 42
N_DEALS = 200

hands_data = []
for deal_id in range(N_DEALS):
    hands = generate_deal(SEED, deal_id)
    # Vary contract types
    contract_types = ['suit', 'high', 'low']
    contract_type = contract_types[deal_id % 3]
    trump = 'H' if contract_type == 'suit' else None
    
    for seat in range(4):
        hand = hands[seat]
        features = get_hand_features(hand, contract_type, trump)
        hands_data.append({
            'hand_id': f"{deal_id}_{seat}",
            'deal_id': deal_id,
            'seat': seat,
            'contract_type': contract_type,
            'trump': trump,
            # Add feat_ prefix for diagnostic charts
            **{f'feat_{k}': v for k, v in features.items()}
        })

df = pd.DataFrame(hands_data)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)[:10]}...")
df.head(3)

In [ ]:
# Generate Dict structure for evaluation charts
features_by_contract = {'suit_H': [], 'high': [], 'low': []}

for deal_id in range(N_DEALS):
    hands = generate_deal(SEED, deal_id)
    
    for contract_key, contract_type, trump in [
        ('suit_H', 'suit', 'H'),
        ('high', 'high', None),
        ('low', 'low', None),
    ]:
        # Just use seat 0 for simplicity
        features = get_hand_features(hands[0], contract_type, trump)
        features_by_contract[contract_key].append(features)

print(f"Dict structure: {list(features_by_contract.keys())}")
print(f"Samples per contract: {len(features_by_contract['suit_H'])}")

---
# Part 1: Diagnostic Charts (DataFrame-based)

From `bid_euchre.diagnostics` — interactive analysis of bidless datasets.

**Input:** pandas DataFrame with `feat_*` columns

## 1.1 `plot_hand_value_by_seat()`

Box plots showing hand_value distribution across seats (0-3). Detects dealing bias or per-seat feature computation bugs.

In [ ]:
fig = plot_hand_value_by_seat(df)
plt.show()

## 1.2 `plot_hand_value_by_contract()`

Box plots comparing hand_value across contract types (suit/high/low).

In [ ]:
fig = plot_hand_value_by_contract(df)
plt.show()

## 1.3 `plot_feature_distributions()`

Grid of histograms for multiple features. Shows top 9 features by variance by default.

In [ ]:
fig = plot_feature_distributions(df)
plt.show()

In [ ]:
# With specific features
fig = plot_feature_distributions(df, features=['hand_value', 'trump_count', 'offsuit_aces'])
plt.show()

## 1.4 `plot_feature_correlation()`

Heatmap of feature correlations. Top 10 features by variance by default.

In [ ]:
fig = plot_feature_correlation(df)
plt.show()

## 1.5 `plot_rolling_mean()`

Time-series plot of rolling mean over hand index. Detects drift over time.

In [ ]:
fig = plot_rolling_mean(df, column='feat_hand_value', window=50)
plt.show()

## 1.6 `plot_feature_vs_label()`

Dual panel: scatter plot + binned box plot. Shows feature vs label relationship.

In [ ]:
fig = plot_feature_vs_label(df, feature='trump_count', label='hand_value')
plt.show()

---
# Part 2: Evaluation Charts (Dict-based)

From `bid_euchre.reporting.validation` — batch report generation for training pipelines.

**Input:** `Dict[contract_key, List[feature_dict]]` or `List[feature_dict]`

**Output:** Saves PNG files to disk, returns file path

In [ ]:
# Create temp directory for output
output_dir = tempfile.mkdtemp(prefix='charts_demo_')
print(f"Output directory: {output_dir}")

## 2.1 `plot_feature_distributions()` (eval)

Feature distributions by contract type. Overlays histograms for comparison.

In [ ]:
path = eval_plot_feature_distributions(
    features_by_contract,
    output_dir,
    feature_keys=["trump_count", "hand_value"],
)
print(f"Saved to: {path}")

# Display the saved image
Image(filename=path)

## 2.2 `plot_feature_correlation()` (eval)

Correlation matrix from feature dicts. Auto-detects numeric columns.

In [ ]:
# Flatten all features for correlation
all_features = []
for features_list in features_by_contract.values():
    all_features.extend(features_list)

path = eval_plot_feature_correlation(all_features, output_dir)
print(f"Saved to: {path}")

Image(filename=path)

## 2.3 `plot_hand_value_by_contract()` (eval)

Box plots of hand_value by contract type from Dict input.

In [ ]:
path = eval_plot_hand_value_by_contract(features_by_contract, output_dir)
print(f"Saved to: {path}")

Image(filename=path)

## 2.4 `generate_validation_plots()`

Orchestrator function that generates all evaluation plots at once.

In [ ]:
# Use a separate subdirectory
batch_dir = os.path.join(output_dir, "batch")

plots = generate_validation_plots(features_by_contract, batch_dir)
print("Generated plots:")
for name, path in plots.items():
    print(f"  {name}: {path}")

# Quick Reference

## Diagnostic Charts (`bid_euchre.diagnostics`)

### Feature Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_hand_value_by_seat(df)` | Seat balance check | `df` with `seat`, `feat_hand_value` |
| `plot_hand_value_by_contract(df)` | Contract comparison | `df` with `contract_type`, `feat_hand_value` |
| `plot_feature_distributions(df)` | Feature histograms | `features=None` for top 9 by variance |
| `plot_feature_correlation(df)` | Correlation heatmap | `features=None` for top 10 by variance |
| `plot_rolling_mean(df, column)` | Drift detection | `window=100` default |
| `plot_feature_vs_label(df, feature)` | Scatter + boxplot | `label='feat_hand_value'` default |

### Distribution Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_cdf(df, column)` | Cumulative distribution | `group_by=None` for overlaid CDFs |
| `plot_ccdf(df, column)` | Tail distribution (1-CDF) | `log_scale=True` for heavy tails |

### Outcome Evaluation Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_feature_vs_outcome(df, feature)` | Feature vs outcome with correlation | `outcome='tricks_won'` |
| `plot_outcome_distributions(df, outcome)` | Outcome by category | `group_by='contract_type'` |
| `plot_feature_outcome_correlation(df)` | Feature importance bar chart | `outcome='tricks_won'`, `top_n=15` |

### Trump Suit Analysis Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_hand_value_by_trump_suit(df)` | Hand value by suit | `show_variance=True` |
| `plot_outcome_by_trump_suit(df)` | Tricks won by suit | `outcome='tricks_won'` |
| `plot_feature_heatmap_by_suit(df)` | Feature means by suit | `normalize=True`, `features=None` |
| `plot_suit_variance_summary(df)` | Variance comparison | `column='feat_hand_value'` |

### Strategy Comparison Charts

| Function | Purpose | Key Parameters |
|----------|---------|----------------|
| `plot_win_rate_heatmap(matchup_results)` | All-vs-all win rate matrix | `metric='win_rate'` |
| `plot_tricks_distribution_comparison(matchup_results)` | Violin plots by matchup | `team=0` |
| `plot_strategy_delta_bars(baseline, comparisons)` | Delta vs baseline | `metric='mean_tricks'` |
| `plot_self_play_control(self_play_results)` | Self-play sanity check | `expected_mean=5.0` |
| `plot_matchup_summary(matchup_results)` | 3-panel summary | Heatmap + bars + control |

## Evaluation Charts (`bid_euchre.reporting.validation`)

| Function | Purpose | Input Format |
|----------|---------|---------------|
| `plot_feature_distributions(fbc, dir)` | By-contract histograms | `Dict[str, List[Dict]]` |
| `plot_feature_correlation(features, dir)` | Correlation matrix | `List[Dict]` |
| `plot_hand_value_by_contract(fbc, dir)` | Contract box plots | `Dict[str, List[Dict]]` |
| `generate_validation_plots(fbc, dir)` | All of the above | `Dict[str, List[Dict]]` |

---
# Part 3: Outcome Evaluation Charts

Charts that correlate hand features with actual outcomes (`tricks_won`) from simulation.

**Key insight:** The bidless dataset only contains input features. To analyze outcomes, we must run simulations that record both starting features AND resulting trick counts.

## 3.0 Generate Simulation Data (features + tricks_won)

Run simulations to create a dataset with both hand features and actual trick outcomes.

In [ ]:
# Generate simulation data with features + tricks_won
from bid_euchre.sim.simulation import play_single_hand
from bid_euchre.strategy import GreedyStrategy

# Run simulations to get outcome data
N_DEALS_SIM = 200
outcome_data = []
strategy = GreedyStrategy()

for deal_id in range(N_DEALS_SIM):
    hands = generate_deal(SEED, deal_id)
    
    for contract_type in ['suit', 'high', 'low']:
        trump = 'H' if contract_type == 'suit' else None
        
        t0, t1, _, all_feats, _, _, *_ = play_single_hand(
            contract_type=contract_type,
            trump_suit=trump,
            strategy=strategy,
            hands=hands,
            deal_seed=SEED,
        )
        
        # Record each player's features + their team's tricks
        for seat in range(4):
            team_tricks = t0 if seat in (0, 2) else t1
            features = all_feats[seat]
            outcome_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': contract_type,
                'trump': trump,
                'tricks_won': team_tricks,
                **{f'feat_{k}': v for k, v in features.items()}
            })

outcome_df = pd.DataFrame(outcome_data)
print(f"Simulation DataFrame shape: {outcome_df.shape}")
print(f"Tricks won range: {outcome_df['tricks_won'].min()} - {outcome_df['tricks_won'].max()}")
outcome_df.head(3)

## 3.1 `plot_feature_vs_outcome()` - hand_value vs tricks_won

Scatter plot with trend line + binned box plot. Shows correlation coefficient in title.

In [ ]:
from bid_euchre.diagnostics.charts import plot_feature_vs_outcome

# hand_value vs tricks_won - the key relationship
fig = plot_feature_vs_outcome(outcome_df, feature='hand_value', outcome='tricks_won')
plt.show()

In [ ]:
# Also check trump_count vs tricks_won for suit contracts
suit_df = outcome_df[outcome_df['contract_type'] == 'suit']
fig = plot_feature_vs_outcome(suit_df, feature='trump_count', outcome='tricks_won')
plt.show()

## 3.2 `plot_outcome_distributions()` - tricks by contract type

Violin/box plots of outcome distribution grouped by category.

In [ ]:
from bid_euchre.diagnostics.charts import plot_outcome_distributions

fig = plot_outcome_distributions(outcome_df, outcome='tricks_won', group_by='contract_type')
plt.show()

## 3.3 `plot_feature_outcome_correlation()` - feature importance bar chart

Horizontal bar chart showing correlation of each feature with tricks_won, sorted by importance.

In [ ]:
from bid_euchre.diagnostics.charts import plot_feature_outcome_correlation

fig = plot_feature_outcome_correlation(outcome_df, outcome='tricks_won', top_n=15)
plt.show()

In [ ]:
# Filter to suit contracts only for more specific feature importance
fig = plot_feature_outcome_correlation(suit_df, outcome='tricks_won', top_n=15)
plt.show()

---
# Part 4: Strategy Comparison Charts

Charts for comparing strategy performance in head-to-head and self-play matchups.

Available strategies:
- **Greedy**: 1-trick lookahead, plays cheapest winning card
- **Glutton**: Partner-aware greedy with trump conservation
- **Random Legal**: Uniform random among legal moves (baseline)

## 4.0 Generate Matchup Data

Run simulations with different strategy pairs to collect matchup results.

In [ ]:
import numpy as np

from bid_euchre.strategy import GluttonStrategy, RandomLegalStrategy

# Define strategies to compare
STRATEGY_CLASSES = {
    'greedy': GreedyStrategy,
    'glutton': GluttonStrategy,
    'random': lambda: RandomLegalStrategy(seed=42),
}

# Generate matchup results
N_HANDS = 200
matchup_results = {}

for team0_name in STRATEGY_CLASSES.keys():
    for team1_name in STRATEGY_CLASSES.keys():
        team0_strat = STRATEGY_CLASSES[team0_name]()
        team1_strat = STRATEGY_CLASSES[team1_name]()
        strategies = [team0_strat, team1_strat, team0_strat, team1_strat]
        
        t0_tricks = []
        t1_tricks = []
        
        for deal_id in range(N_HANDS):
            hands = generate_deal(SEED, deal_id)
            t0, t1, *_ = play_single_hand(
                contract_type='suit',
                trump_suit='H',
                strategies=strategies,
                hands=hands,
                deal_seed=SEED,
            )
            t0_tricks.append(t0)
            t1_tricks.append(t1)
        
        matchup_results[(team0_name, team1_name)] = {
            'tricks_team0': t0_tricks,
            'tricks_team1': t1_tricks,
            'mean_tricks': np.mean(t0_tricks),
            'win_rate': np.mean([1 if t >= 6 else 0 for t in t0_tricks]),
            'ci_lower': np.mean(t0_tricks) - 1.96 * np.std(t0_tricks) / np.sqrt(N_HANDS),
            'ci_upper': np.mean(t0_tricks) + 1.96 * np.std(t0_tricks) / np.sqrt(N_HANDS),
        }

print(f"Generated {len(matchup_results)} matchups")
for key, result in matchup_results.items():
    print(f"  {key[0]} vs {key[1]}: mean={result['mean_tricks']:.2f}, win_rate={result['win_rate']:.1%}")

## 4.1 `plot_win_rate_heatmap()` - all-vs-all win rates

Heatmap showing Team 0's win rate against each Team 1 strategy.

In [ ]:
from bid_euchre.diagnostics import plot_win_rate_heatmap

fig = plot_win_rate_heatmap(matchup_results, metric='win_rate')
plt.show()

## 4.2 `plot_tricks_distribution_comparison()` - violin plots by matchup

Violin plots comparing trick distributions across different matchups.

In [ ]:
from bid_euchre.diagnostics import plot_tricks_distribution_comparison

# Show a subset of interesting matchups
subset_matchups = {k: v for k, v in matchup_results.items() 
                   if k[0] != k[1]}  # Exclude self-play for comparison

fig = plot_tricks_distribution_comparison(subset_matchups, team=0)
plt.show()

## 4.3 `plot_strategy_delta_bars()` - delta vs baseline

Bar chart showing mean tricks delta relative to baseline (random).

In [ ]:
from bid_euchre.diagnostics import plot_strategy_delta_bars

# Compare each strategy vs random baseline
baseline_results = matchup_results[('random', 'random')]
comparison_results = {
    'greedy': matchup_results[('greedy', 'random')],
    'glutton': matchup_results[('glutton', 'random')],
}

fig = plot_strategy_delta_bars(baseline_results, comparison_results, baseline_name='random')
plt.show()

## 4.4 `plot_self_play_control()` - self-play sanity check

Control chart showing mean tricks for self-play matchups. Should be ~5.0 for fair play.

In [ ]:
from bid_euchre.diagnostics import plot_self_play_control

# Extract self-play matchups
self_play_results = {k[0]: v for k, v in matchup_results.items() if k[0] == k[1]}

fig = plot_self_play_control(self_play_results)
plt.show()

---
# Part 5: Trump Suit Analysis

Charts comparing distributions and outcomes across trump suits (C, D, H, S).

**Key question**: Do certain trump suits lead to systematically different hand strengths or outcomes?

## 5.0 Generate Multi-Suit Data

Generate data with all 4 trump suits for comparison.

In [ ]:
# Generate data with all 4 trump suits
multi_suit_data = []
multi_suit_outcome_data = []

for deal_id in range(N_DEALS_SIM):
    hands = generate_deal(SEED, deal_id)
    
    for trump in ['C', 'D', 'H', 'S']:
        # Features only (no simulation)
        for seat in range(4):
            features = get_hand_features(hands[seat], 'suit', trump)
            multi_suit_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': 'suit',
                'trump': trump,
                **{f'feat_{k}': v for k, v in features.items()}
            })
        
        # With simulation outcomes
        t0, t1, _, all_feats, _, _, *_ = play_single_hand(
            contract_type='suit',
            trump_suit=trump,
            strategy=strategy,
            hands=hands,
            deal_seed=SEED,
        )
        
        for seat in range(4):
            team_tricks = t0 if seat in (0, 2) else t1
            features = all_feats[seat]
            multi_suit_outcome_data.append({
                'deal_id': deal_id,
                'seat': seat,
                'contract_type': 'suit',
                'trump': trump,
                'tricks_won': team_tricks,
                **{f'feat_{k}': v for k, v in features.items()}
            })

multi_suit_df = pd.DataFrame(multi_suit_data)
multi_suit_outcome_df = pd.DataFrame(multi_suit_outcome_data)

print(f"Multi-suit DataFrame: {multi_suit_df.shape}")
print(f"Trump distribution: {multi_suit_df['trump'].value_counts().to_dict()}")

## 5.1 `plot_hand_value_by_trump_suit()` - hand value by suit

Box plots showing hand_value distribution for each trump suit. Includes variance annotations.

In [ ]:
from bid_euchre.diagnostics import plot_hand_value_by_trump_suit

fig = plot_hand_value_by_trump_suit(multi_suit_df, show_variance=True)
plt.show()

## 5.2 `plot_outcome_by_trump_suit()` - tricks won by suit

Outcome distribution showing if some suits systematically win more tricks.

In [ ]:
from bid_euchre.diagnostics import plot_outcome_by_trump_suit

fig = plot_outcome_by_trump_suit(multi_suit_outcome_df, outcome='tricks_won')
plt.show()

## 5.3 `plot_feature_heatmap_by_suit()` - feature means by suit

Heatmap showing which features vary most across trump suits.

In [ ]:
from bid_euchre.diagnostics import plot_feature_heatmap_by_suit

fig = plot_feature_heatmap_by_suit(multi_suit_df, normalize=True)
plt.show()

## 5.4 `plot_suit_variance_summary()` - variance comparison

Bar chart comparing variance of hand_value across suits.

In [ ]:
from bid_euchre.diagnostics import plot_suit_variance_summary

fig = plot_suit_variance_summary(multi_suit_df, column='feat_hand_value')
plt.show()

---
# Part 6: Distribution Analysis Charts

CDF and CCDF plots for analyzing distribution shapes and tail behavior.

**CDF (Cumulative Distribution Function)**: Shows P(X ≤ x) — the probability a value is less than or equal to x.
- Useful for understanding distribution shape
- Quartile reference lines show median and spread

**CCDF (Complementary CDF)**: Shows P(X > x) — the probability a value exceeds x.
- Log scale reveals tail behavior
- Useful for identifying rare high-value hands

## 6.1 `plot_cdf()` - Cumulative Distribution Function

CDF for hand_value showing probability distribution shape with quartile markers.

In [ ]:
from bid_euchre.diagnostics import plot_cdf

# Basic CDF of hand_value
fig = plot_cdf(df, column='feat_hand_value')
plt.show()

In [ ]:
# CDF grouped by contract_type - compare distributions across categories
fig = plot_cdf(df, column='feat_hand_value', group_by='contract_type')
plt.show()

## 6.2 `plot_ccdf()` - Complementary CDF (Tail Analysis)

CCDF with log scale reveals tail behavior — useful for identifying rare high-value hands.

In [ ]:
from bid_euchre.diagnostics import plot_ccdf

# CCDF of hand_value with log scale - reveals tail distribution
fig = plot_ccdf(df, column='feat_hand_value', log_scale=True)
plt.show()

## 6.3 CDF of Tricks Won

CDF of simulation outcomes — shows probability of winning ≤ N tricks.

In [ ]:
# CDF of tricks_won - uses outcome_df from Part 3
fig = plot_cdf(outcome_df, column='tricks_won', group_by='contract_type')
plt.show()

## 6.4 CCDF of Tricks Won

CCDF shows P(tricks > N) — useful for analyzing win probability thresholds.

In [ ]:
# CCDF of tricks_won - P(tricks > N) by contract type
# At x=5, the CCDF shows win probability (≥6 tricks)
fig = plot_ccdf(outcome_df, column='tricks_won', log_scale=False, group_by='contract_type')
plt.show()

In [ ]:
# Cleanup temp directory
shutil.rmtree(output_dir)
print("Cleaned up temp directory")